# MN5162 Section 3 Assignment

**Student Name:** Stephen White<br>
**Student ID:** 13159178

## Topic 1

### Current State:
An element of our offering surrounds the classification of carrier tracking events to a predefined structured set of events for our customers, e.g. in transit, out for delivery, delivered, attempted delivery, exception, etc. Beyond providing a useful point in time representation of where a customer's parcel is in its journey to the ultimate end user of its products, these categorisations are also used in driving customer communications such as "your parcel is out for delivery", with the attendant marketing opportunities you could expect. Historically, this mapping exercise has been a manual process that is maintained within the codebase. With varying degrees of complexity, for the most part carriers provided structured mapping files which may contain an event type id and typical or boilerplate tracking description of the event, such as "on route from depot X to depot Y" for example. Given these data points, we would classify the event as being "in transit" and future tracking events received matching these characteristics are labelled as such. As could be expected, this external data can change over time as new event ids are added, new descriptions are provided, or indeed a given event type id may be re-classified at source for various reasons. All of which require codebase maintenance and changes, and can lead to incorrect classifications when such changes are not informed of beforehand. Moreover, this process is unique to each carrier within our network and to date we have not incorporated an ability to classify unseen events based on this labelled dataset, either for unseen events of carriers in our network, or for new integrations with carriers outside the network.

In summary, we maintain a manual, rules-based system for classifying carrier tracking events into standardized categories. This results in:
- manual codebase maintenance for additions or updates to carrier provided data.
- separate classification logic for each carrier in the network.
- an inability to classify unseen tracking descriptions, either for existing or prospective carriers.
- reactive fixes when carriers update event definitions, leading to temporary (if caught) misclassifications.

### Impact:
- Operational overhead from development effort spent on carrier-specific rule updates.
- Accuracy concerns stemming from delayed awareness of changes.
- Scalability concerns around the substantial engineering effort required for new carrier integrations.
- Limited innovation with the inability to improve automatically from new data.

### Proposal: Transformer-based event classification:
The proposed approach would leverage pre-trained transformer models to automatically classify tracking events into defined categories. Benefitting from transfer learning, such models capture the semantic meaning and context from text. Depending on scope, there exists the capacity to train or fine-tune such models on the existing labelled dataset of carrier tracking description strings and label pairs. Moreover, the transition between states, such as in transit to out for delivery, could be encoded and appended to the feature set. The transformer approach would provide zero-shot capability against unseen carrier descriptions and a level of generalization that would facilitate the use of a single model to classify across multiple carriers once fine tuned on diverse examples.

### Module Experience:
In the assignment for section 1, I implemented sentiment analysis on customer product reviews using a bert-base-multilingual-uncased model finetuned for sentiment analysis on product reviews which predicts the sentiment of the review as a number of stars (between 1 and 5). The approach mirrors what would be needed for tracking event classification whereby now the set of labels would include IN_TRANSIT, OUT_FOR_DELIVERY, etc. rather than POSTIVE/NEGATIVE, or 3_STARS, 4_STARS, etc. The performance of that fined tuned model within its chosen domain of product reviews was quite impressive and implies a similar performance could be achieved in our use case. The approach followed in that case was:

Using: https://huggingface.co/nlptown/bert-base-multilingual-uncased-sentiment

"This is a bert-base-multilingual-uncased model finetuned for sentiment analysis on product reviews in six languages: English, Dutch, German, French, Spanish, and Italian. It predicts the sentiment of the review as a number of stars (between 1 and 5)."

In [ ]:
from transformers import pipeline

In [ ]:
sentiment_analyzer = pipeline(
    task='sentiment-analysis',
    model='nlptown/bert-base-multilingual-uncased-sentiment',
    max_length=512,
    truncation=True
)

In [ ]:
def stars_to_sentiment(label):
    stars = int(label.split()[0])
    return 'negative' if stars < 3 else 'neutral' if stars == 3 else 'positive'

In [ ]:
for letter in letters:
    scores = sentiment_analyzer(letter['text'])
    label = max(scores, key=lambda x: x['score'])
    letter['predicted_rating'] = label['label']
    letter['sentiment_score'] = label['score']
    letter['sentiment_label'] = stars_to_sentiment(label['label'])

With the necessary alterations to the above example snippets, an equivalent transformer model ought to learn the semantic relationships between event descriptions and their categorisation, enabling classification of previoulsy unseen descriptions from new carriers or modified descriptions from existing carriers. It could be expected that transformer-based models could achieve high accuracy on diverse, unstructured text of varied length. Indeed, it is likely that the nature of tracking descriptions, which tend to be of a certain uniform language style and length, would facilitate high accuracy.

The **benefits** associated with this approach include: 
- reduced maintenance for carrier updates.
- faster integration with new carriers.
- improved accuracy as the automated system reduces human error in classification maintenance.
- scalability as a single model can serve a growing carrier network.
- a continuous feedback look can improve the model as new patterns emerge.

The **costs** associated include the development expense from POC, data preparation in structuring appropriate labelled datasets, deploying inference in the production tracking pipeline, and ongoing model monitoring and periodic retraining.

The **risks** associated include possible distribution shift if carrier descriptions change significantly (unlikely, and covered with retraining), unusual or ambiguous edge cases with low confidence scores, and direct dependency on the training data quality.

In mitigating these risks, it is recommended to implement confidence score thresholds with human feedback incorporated, to maintain a hybrid system initially for comparison, and to establish clear and extensive monitoring.

## Topic 2

### Current State:
One such tracking event mapping is that to "exception"; indicating a deviation from the happy path for a parcel. A wide array of impediments may give rise to exceptions, such as delay in transit due to a breakdown, a parcel being held at customs awaiting payment, damage or loss in transit, etc. Exceptions are typically viewed as high-touch events for our customers that require proactive remediation for their end customers in a timely fashion. Given such, our customers require detailed information about such events in order to engage in their resolution, noting again that a variety of issues lead to such exceptions, but these reasons often cluster nicely into a somewhat narrow set of labels. We do not possess a labelled dataset for a supervised approach, but unsupervised methods can yield the exception sub-statuses that we require.

In summary, we receive diverse exception descriptions from carriers but lack an automated process to consistently categorize them. We lack a standardized taxonomy of exception sub-statuses and a labelled dataset for supervised learning. At an extreme, customer service operatives are required to decipher carrier tracking descriptions of exception events in order to assign labels corresponding to their cause and required remediation. 

### Impact:
- Operational burden on customer service teams.
- Inconsistent treatment depending on which agent categorizes a given event.
- Unknown root causes of exceptions.
- No clear picture of exception categories affecting health of the network.

### Proposal: Combination of Summarization and Unsupervised Clustering:
The proposed approach considers the use of summarization using transformer models to compress verbose exception descriptions to key concepts, and the subsequent clustering of such summarized text to identify recurring patterns and discover exception sub-statuses. This approach should discover natural clusters of similar exception types without the need for labelled data.

### Module Experience:
In the assignment for section 1, I implemented text summarisation using a pre-trained BART transformer model:

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

In [ ]:
def make_summarizer(model_id="facebook/bart-large-cnn"):
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

    def summarize(text, max_new_tokens=80):
        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=1024,
        )
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                num_return_sequences=1,
            )
        summary = tokenizer.decode(out[0], skip_special_tokens=True)
        return summary

    return summarize

In [ ]:
summarize = make_summarizer()

In [ ]:
for letter in letters:
    letter['summary'] = summarize(letter['text'], max_new_tokens=256)

In doing so, an example customer review:
```
'text': "I used to buy the Costco socks that were a high content Merino wool. Last year, they changed the ingredients to be less wool and more recycled fibres, which essentially felt like trash on my hands (I shudder to imagine what hiking in those trash socks would feel like, but I'm guessing it'd be horrid). Defeated after the annual Costco sock run (I'd gone to costco specifically to treat myself to a new pair of socks-a tradition I indulge in every year, much to the chagrin of my burgeoning sock drawer). So...I did the next logical thing and took to Amazon looking for a high percentage wool sock that would meet my needs... i.e. keeping my feet cosy and warm. These socks fit the bill and then some. Not only are they a high percentage wool, one of the highest available on the Internet to buy, they're also an amazing price. Upon putting the socks on my feet, I realised that this was no ordinary wool sock. These socks were softer and more durable than even the Costco socks, which I've been obsessed with for many years. The real test happens in the wash. You see, I'm rather lazy and never bother to separate out the regular washes. Everything goes in at the same time and goes into the dryer for a good hot tumble round. These socks were unaffected by this, remained soft and cosy, and have continued to outperform any sock I've ever owned. I'm writing this review about 1 year after buying. They remain in near new condition in terms of softness, warmth, and durability. No holes or irritating seams, or weird shrinkages to be found. They fit well to the foot. I've put these socks through the paces, as I wear wool socks year round, due to my ridiculously cold feet. Hiking, farm work, gardening, camping, and walking around in them have not done a thing to them. Coming from a sock connoisseur who's done an immense amount of research, these socks are it. So much so that I'm coming back here, a year later, to write this review and buy more. So if you're looking for a sign....here it is. Buy these socks. You won't regret it. Buy them for your adult friends....because nothing beats getting a high quality and high wool content sock.",
```

was compressed to:
```
'summary': 'Sock is high percentage wool, one of the highest available on the Internet to buy. They remain in near new condition in terms of softness, warmth, and durability. No holes or irritating seams, or weird shrinkages to be found. Hiking, farm work, gardening, camping, and walking around in them have not done a thing to them.'
```

For carrier exception descriptions, the same approach would summarise verbose exception descriptions and extract key concepts. An unsupervised clustering exercise on these summaries should then cluster similar summaries to discover natural groupings. Exemplars from within identified clusters can then be used to derive a labelled dataset of exception sub-statuses, such as CUSTOMS_HOLD, for example. This then enables an automated root cause extraction identifying the primary reason for an exception from its tracking description, a consistent treatment across exceptions, and actionable information for customer service teams.

The **benefits** associated with this approach include:
- automated discovery of exception patterns without human labelling.
- consistency as similar exceptions receive the same sub-statuses.
- extracted key reasoning enables personalized customer communication
- scalability as the model handles a growing exception volume.
- a clear picture of exception distributions across the network.
- low training cost as the unsupervised approach requires no labelled data.

The **costs** associated include the initial development expense to establish the pipeline, a compute-intensive summarisation step, required infrastructure for inference in production, and human feedback and validation expense.

The **risks** associated include summarization quality, cluster ambiguity that may not align with business categorisation, and latency particularly from the summarization step.

In mitigating these risks, it is recommended to compare original descriptions with summarized output establishing an assessment approach that leverages appropriate evaluation metrics, incorporate domain expertise in the validation of discovered sub-statuses, plan for the required asychronous deployment, and monitor cluster stability over time, retraining as required.